# 05 — Correlaciones

**Librerias usadas:** pandas, seaborn, matplotlib, scipy

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'solineras'))
os.chdir(ROOT)
DATA = ROOT / 'data'

f = DATA / 'clean' / 'encuestas_consolidadas.parquet'
if not f.exists():
    from consolidar import consolidar
    consolidar()
df = pd.read_parquet(f)
print('Dataset:', df.shape)

## 1. Matriz de correlacion

In [ ]:
cols = ['potencia_pico_kwp', 'num_paneles', 'capacidad_baterias_kwh',
        'consumo_total_anual_kwh', 'pr_promedio', 'satisfaccion',
        'kwh_por_kwp', 'antiguedad_anios', 'ratio_bateria_panel']
cols = [c for c in cols if c in df.columns and df[c].notna().sum() >= 3]

if len(cols) >= 2:
    sub = df[cols].dropna(thresh=2)
    corr = sub.corr(method='spearman').round(3)
    display(corr)
else:
    print('Insuficientes columnas numericas')

## 2. Heatmap

In [ ]:
if len(cols) >= 2:
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f',
                square=True, cbar_kws={'shrink': 0.8})
    plt.title('Correlaciones Spearman')
    plt.tight_layout()
    plt.show()

## 3. Pares significativos

In [ ]:
if len(cols) >= 2:
    pares = []
    for i, a in enumerate(cols):
        for b in cols[i+1:]:
            s2 = df[[a, b]].dropna()
            if len(s2) >= 3:
                r, p = spearmanr(s2[a], s2[b])
                pares.append({'var_a': a, 'var_b': b,
                              'rho': round(r, 3), 'p': round(p, 4),
                              'n': len(s2), 'significativo': p < 0.05})
    if pares:
        display(pd.DataFrame(pares).sort_values('rho', key=abs, ascending=False).head(15))